# 04 — Gamma-or-Skip Development Evaluation

Evaluate this run's selected detector on its own policy's development (test) split with native FP32 Ultralytics validation, saving per-image matches for dark/bright and Apply/Skip cohorts.

**Inputs:** runs/v1-YYYY-MM-DD/train/weights/best.pt and the run's prepared data.

**Outputs:** runs/v1-YYYY-MM-DD/eval/{metrics.json,paired_stats.npz,plots} only when ALLOW_INFERENCE=True.

**Prerequisites:** Completed 02 and 03. Evaluation never imports or invokes training.

**Outline**

- [1. Configuration](#section-1)
- [2. Policy–Detector Pair](#section-2)
- [3. Evaluation Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

RUN_DATE=None uses the newest run. Saved metrics are reused when inputs match.

In [1]:
import os

import pandas as pd
from IPython.display import display

from vehicle_detection.common import project_root, require_paths
from vehicle_detection.evaluation import evaluate_experiment
from vehicle_detection.experiment_data import load_experiment

ROOT = project_root()
os.chdir(ROOT)  # Run All behaves the same from any kernel working directory.
EXPERIMENT = "v1"
require_paths(
    config=ROOT / f"notebooks/experiments/{EXPERIMENT}/{EXPERIMENT}.yaml",
    shared=ROOT / "configs/data.yaml",
)
config = load_experiment(ROOT, EXPERIMENT)

RUN_DATE = None
ALLOW_INFERENCE = False
display(pd.Series(config["evaluation"], name="evaluation"))

imgsz             640
batch              32
device              0
workers             0
conf            0.001
iou               0.7
max_det           300
quantize           32
nms             False
agnostic_nms    False
rect             True
augment         False
plots            True
verbose         False
Name: evaluation, dtype: object

<a id="section-2"></a>

## 2. Policy–Detector Pair

Only development is evaluated; the final holdout is never read.

In [2]:
if not ALLOW_INFERENCE:
    print(
        "Detector inference disabled; inspecting saved artifacts only.",
        "Set ALLOW_INFERENCE=True to opt in.",
    )
try:
    result = evaluate_experiment(ROOT, EXPERIMENT, RUN_DATE, allow_inference=ALLOW_INFERENCE)
except FileNotFoundError as error:
    result = None
    print(error)
if result:
    display(pd.DataFrame([result["overall"]]))
    display(pd.DataFrame(result["classes"]).set_index("Class"))

Ultralytics 8.4.198 🚀 Python-3.11.14 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16284MiB)
YOLO26n summary (fused): 122 layers, 2,390,707 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3494.7±1838.3 MB/s, size: 87.4 KB)
val: Scanning /home/truong51972/projects/vehicle-detection/runs/v1-2026-11-09/data/labels/test... 13975 images, 121 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 13975/13975 1.3Kit/s 10.8s0.1s
val: New cache created: /home/truong51972/projects/vehicle-detection/runs/v1-2026-11-09/data/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 437/437 3.9it/s 1:510.2sss
                   all      13975     152774      0.558      0.394      0.417      0.254
Speed: 0.2ms preprocess, 1.3ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /home/truong51972/projects/vehicle-detection/runs/v1-2026-11-09/eval


,precision,recall,mAP50,mAP50_95
0,0.557908,0.393709,0.417177,0.254374


,Images,Instances,Box-P,Box-R,Box-F1,mAP50,mAP75,mAP50-95
Class,,,,,,,,
car,13820,142541,0.691342,0.673084,0.682091,0.717444,0.419995,0.425435
truck,3773,5965,0.538875,0.445432,0.487718,0.457116,0.349481,0.318501
bus,1759,2276,0.515401,0.431898,0.469969,0.430409,0.360929,0.319843
motorcycle,435,562,0.501480,0.186833,0.272239,0.206733,0.054414,0.089514
bicycle,872,1430,0.542440,0.231298,0.324310,0.274182,0.081663,0.118577


<a id="section-3"></a>

## 3. Evaluation Summary

Cohort AP keeps every GT of the selected images; dark means raw brightness < 0.12.

In [3]:
if result:
    display(
        pd.DataFrame(
            {
                k: {"images": v["images"], "mAP50_95": v["mAP50_95"]}
                for k, v in result["cohorts"].items()
            }
        ).T
    )
    print("Run:", result["inputs"]["run"], "| speed (ms):", result["speed_ms"])
print("Final holdout stays sealed.")

,images,mAP50_95
dark,3307.0,0.227799
bright,10668.0,0.259344
apply,3307.0,0.227799
skip,10668.0,0.259344


Run: runs/v1-2026-11-09 | speed (ms): {'preprocess': 0.21051740400715513, 'inference': 1.3325805411090874, 'loss': 0.00024408100179722114, 'postprocess': 0.20906366847944455}
Final holdout stays sealed.
